$$
m_k = (k-1)\sigma^2 m_{k-2} + \mu m_{k-1} - \sigma \frac{b^{k-1}\phi\left(\frac{b-\mu}{\sigma}\right) - a^{k-1}\phi\left(\frac{a-\mu}{\sigma}\right)}{\Phi\left(\frac{b-\mu}{\sigma}\right) - \Phi\left(\frac{a-\mu}{\sigma}\right)}, \quad k = 1, 2, \ldots,
$$

$m_{-1}=0$ and $m_0=1$.

In [1]:
using Distributions

In [2]:
ϕ(z) = pdf(Normal(), z)
Φ(z) = cdf(Normal(), z);

In [3]:
mm(m_prev, m_prev2, k, μ, σ; a=0, b=1) = (k-1)*σ^2*m_prev2 + μ*m_prev - σ*(b^(k-1)*ϕ((b-μ)/σ)-a^(k-1)*ϕ((a-μ)/σ)) / (Φ((b-μ)/σ)-Φ((a-μ)/σ));

In [4]:
mm(1, -1, 1, 0.7, 0.3)

0.6221619232342932

In [5]:
mean(truncated(Normal(0.7, 0.3), 0, 1))

0.6221619232342932

In [6]:
function m_sequence(μ, σ; n = 10)
    m_array = Vector{Float64}(undef, n)
    m_array[1] = mm(1, -1, 1, μ, σ)
    m_array[2] = mm(m_array[1], 1, 2, μ, σ)
    for i in 3:n
        m_array[i] = mm(m_array[i-1], m_array[i-2], i, μ, σ)
    end
    return m_array
end;

In [59]:
# this method of m_sequence is specified the first and second moment
function m_sequence(μ, σ, m1, m2; n = 10)
    m_array = Vector{Float64}(undef, n)
    m_array[1] = m1
    m_array[2] = m2
    for i in 3:n
        m_array[i] = mm(m_array[i-1], m_array[i-2], i, μ, σ)
    end
    return m_array
end;

# quick dirty check

In [60]:
m_seq = m_sequence(0.5,40; n = 3) #almost uniform

3-element Vector{Float64}:
 0.5
 0.3333315972352011
 0.249997395852688

In [61]:
m_sequence(0.5,40, m_seq[1], m_seq[2]; n = 3)

3-element Vector{Float64}:
 0.5
 0.3333315972352011
 0.249997395852688

In [11]:
m_seq[2] - m_seq[1]^2

0.08333159723520112

In [12]:
1/12 

0.08333333333333333

In [13]:
m_seq = m_sequence(0.5,5; n = 4)

4-element Vector{Float64}:
 0.5
 0.33322227516749336
 0.24983341275124005
 0.19980961910511752

In [14]:
using Statistics

In [16]:
data = rand(truncated(Normal(0.5,5), 0,1),10^7);
[mean((x)->x^k, data) for k in 1:4]

4-element Vector{Float64}:
 0.4998911844466748
 0.3331031671241842
 0.24971483364508623
 0.19969502139829828

## Complete monotonicity

A sequence $\{a_n\}_{n=0}^{\infty}$ is completely monotone if $(-1)^k\Delta^k a_n \geq 0$ for all $k \geq 0$ and $n \geq 0$, where $\Delta^k$ denotes the $k$-th forward difference operator.

## Specific Cases

For $k = 0$: The sequence is non-negative.
$$\Delta^0 a_n = a_n \geq 0$$

For $k = 1$: The sequence is decreasing.
$$\Delta^1 a_n = a_n - a_{n+1} \geq 0$$

For $k = 2$: The sequence is convex.
$$\Delta^2 a_n = a_n - 2a_{n+1} + a_{n+2} \geq 0$$

For $k = 3$: The third difference is non-positive.
$$\Delta^3 a_n = a_n - 3a_{n+1} + 3a_{n+2} - a_{n+3} \leq 0$$

## General Formula

For $k \geq 4$ (and actually for all $k$), the $k$-th difference can be expressed using binomial coefficients:

$$\Delta^k a_n = \sum_{i=0}^{k} (-1)^i \binom{k}{i} a_{n+i}$$

A sequence is completely monotone if and only if $(-1)^k\Delta^k a_n \geq 0$ for all $k \geq 0$ and $n \geq 0$.

In [17]:
m_seq = m_sequence(0.7,1.2; n = 10)

10-element Vector{Float64}:
 0.5113049157853494
 0.3427780235436191
 0.2573673564362142
 0.2058227937076591
 0.17137651116182973
 0.1467522550023006
 0.128284217433694
 0.1139262651206503
 0.10244715291448458
 0.09306198549764155

In [18]:
# first order
k = 1;
[sum((-1)^i * binomial(k, i) * m_seq[n+i] for i in 0:k) for n in 1:length(m_seq)-k]

9-element Vector{Float64}:
 0.16852689224173034
 0.08541066710740486
 0.05154456272855512
 0.034446282545829376
 0.024624256159529123
 0.0184680375686066
 0.01435795231304371
 0.01147911220616571
 0.009385167416843032

In [19]:
# check it
raw_diff_seq(seq) = seq[1:end-1]- seq[2:end]
raw_diff_seq(m_seq)

9-element Vector{Float64}:
 0.16852689224173034
 0.08541066710740486
 0.05154456272855512
 0.034446282545829376
 0.024624256159529123
 0.0184680375686066
 0.01435795231304371
 0.01147911220616571
 0.009385167416843032

In [20]:
# second order
k = 2;
[sum((-1)^i * binomial(k, i) * m_seq[n+i] for i in 0:k) for n in 1:length(m_seq)-k]

8-element Vector{Float64}:
 0.08311622513432548
 0.03386610437884974
 0.017098280182725745
 0.009822026386300253
 0.006156218590922524
 0.00411008525556289
 0.002878840106878
 0.002093944789322677

In [21]:
# check it
raw_diff_seq(raw_diff_seq(m_seq))

8-element Vector{Float64}:
 0.08311622513432548
 0.03386610437884974
 0.017098280182725745
 0.009822026386300253
 0.006156218590922524
 0.00411008525556289
 0.002878840106878
 0.002093944789322677

In [22]:
# third order
k = 3;
[sum((-1)^i * binomial(k, i) * m_seq[n+i] for i in 0:k) for n in 1:length(m_seq)-k]

7-element Vector{Float64}:
 0.049250120755475746
 0.016767824196123993
 0.007276253796425491
 0.003665807795377729
 0.0020461333353596345
 0.0012312451486848897
 0.000784895317555323

In [23]:
# check it
raw_diff_seq(raw_diff_seq(raw_diff_seq(m_seq)))

7-element Vector{Float64}:
 0.049250120755475746
 0.016767824196123993
 0.007276253796425491
 0.003665807795377729
 0.0020461333353596345
 0.0012312451486848897
 0.000784895317555323

In [24]:
k = 4;
[sum((-1)^i * binomial(k, i) * m_seq[n+i] for i in 0:k) for n in 1:length(m_seq)-k]

6-element Vector{Float64}:
 0.03248229655935175
 0.009491570399698501
 0.0036104460010477624
 0.0016196744600180946
 0.0008148881866747448
 0.0004463498311295666

In [25]:
raw_diff_seq(raw_diff_seq(raw_diff_seq(raw_diff_seq(m_seq))))

6-element Vector{Float64}:
 0.03248229655935175
 0.009491570399698501
 0.0036104460010477624
 0.0016196744600180946
 0.0008148881866747448
 0.0004463498311295666

#### Try uniform 

In [30]:
d_test = Uniform()

Uniform{Float64}(a=0.0, b=1.0)

In [32]:
# Example: moments of the uniform distribution on [0,1] (μ_n = 1/(n+1))
uni_moments = [1/(n+1) for n in 0:8]

9-element Vector{Float64}:
 1.0
 0.5
 0.3333333333333333
 0.25
 0.2
 0.16666666666666666
 0.14285714285714285
 0.125
 0.1111111111111111

In [29]:
data = rand(d_test,10^7);
[mean((x)->x^k, data) for k in 0:8]

9-element Vector{Float64}:
 1.0
 0.5000665361923687
 0.33337398602814483
 0.2500251476209989
 0.20001721446797277
 0.1666792697335976
 0.14286650785700317
 0.12500673832306702
 0.11111555747391007

In [33]:
# third order
k = 3;
[sum((-1)^i * binomial(k, i) * m_seq[n+i] for i in 0:k) for n in 1:length(uni_moments)-k]

6-element Vector{Float64}:
 0.049250120755475746
 0.016767824196123993
 0.007276253796425491
 0.003665807795377729
 0.0020461333353596345
 0.0012312451486848897

In [34]:
function difference_sequence(sequence, k)
    diffs = []
    nmax = length(sequence) - k
    for n in 0:(nmax-1)
        val = 0.0
        for i in 0:k
            val += (-1)^i * binomial(k, i) * sequence[n+i+1]
        end
        push!(diffs, val)
    end
    return diffs
end

# Example: moments of the uniform distribution on [0,1] (μ_n = 1/(n+1))
moments = [1/(n+1) for n in 0:8]

for k in 0:4
    diffs = difference_sequence(moments, k)
    println("k = $k: ", diffs)
    println("Signs: ", [sign(d) for d in diffs])
end

k = 0: Any[1.0, 0.5, 0.3333333333333333, 0.25, 0.2, 0.16666666666666666, 0.14285714285714285, 0.125, 0.1111111111111111]
Signs: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
k = 1: Any[0.5, 0.16666666666666669, 0.08333333333333331, 0.04999999999999999, 0.033333333333333354, 0.023809523809523808, 0.01785714285714285, 0.013888888888888895]
Signs: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
k = 2: Any[0.3333333333333333, 0.08333333333333337, 0.033333333333333326, 0.016666666666666635, 0.009523809523809545, 0.005952380952380959, 0.003968253968253954]
Signs: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
k = 3: Any[0.25, 0.04999999999999999, 0.016666666666666746, 0.007142857142857062, 0.0035714285714285587, 0.0019841269841270326]
Signs: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
k = 4: Any[0.2, 0.033333333333333354, 0.009523809523809657, 0.0035714285714285587, 0.0015873015873015817]
Signs: [1.0, 1.0, 1.0, 1.0, 1.0]


# Optimistic

In [51]:
function diff_check_k(seq, k)
    [sum((-1)^i * binomial(k, i) * seq[n+i] for i in 0:k) for n in 1:length(seq)-k]
end

diff_check_k (generic function with 2 methods)

In [78]:
μ, σ = 0.5, 1.5
d = truncated(Normal(μ, σ), 0, 1)
m1 = mean(d)
m2 = var(d) + m1^2
m_seq = m_sequence(μ, σ, m1, m2; n = 10)

10-element Vector{Float64}:
 0.5
 0.3321053451085569
 0.24815801766283485
 0.19789543342273275
 0.16447522078543653
 0.14066658150701805
 0.12285411646545841
 0.10903106207682
 0.0979949725252176
 0.08898183842677021

In [79]:
m_sequence(μ, σ; n = 10)

10-element Vector{Float64}:
 0.5
 0.33210534510855627
 0.2481580176628344
 0.19789543342272875
 0.1644752207854303
 0.1406665815069701
 0.1228541164653505
 0.10903106207601043
 0.09799497252287015
 0.08898183840920248

In [93]:
μ, σ = 0.5, 1.5 
d = truncated(Normal(μ, σ), 0, 1)
m1 = mean(d)
m2 = var(d) + m1^2
m_seq = m_sequence(μ, σ, m1, m2; n = 20)
diff_check_k(m_seq, 4)

16-element Vector{Float64}:
 0.03342021263728712
 0.009611573358885961
 0.003615399121909224
 0.0016086355380813266
 0.0008043177680523428
 0.0004384364320042522
 0.00025549574174066336
 0.00015703722000859344
 0.00010081918207305307
 6.712549702569959e-5
 4.607918462751215e-5
 3.260995640319564e-5
 2.3369937495409943e-5
 2.194277552058921e-5
 1.3870438304142141e-5
 0.00021173489209447283